Run the cells in order. The first run downloads the GloVe vectors (~66 MB) and caches them in `~/.cache/word_embeddings_demo`.

The only dependency is `numpy`.

In [ ]:
#@title Load the GloVe word vectors
import gzip
import urllib.request
from pathlib import Path

import numpy as np

# GloVe (Wikipedia + Gigaword, 50 dimensions) 
URL = "https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-wiki-gigaword-50/glove-wiki-gigaword-50.gz"
path = Path.home() / ".cache" / "word_embeddings_demo" / "glove-wiki-gigaword-50.gz"

if not path.exists():
    path.parent.mkdir(parents=True, exist_ok=True)
    part = path.with_name(path.name + ".part")
    urllib.request.urlretrieve(URL, part)
    part.replace(path)

with gzip.open(path, "rt", encoding="utf8") as f:
    n_words, dim = map(int, f.readline().split())
    words = []
    vectors = np.empty((n_words, dim), dtype=np.float32)
    for i, line in enumerate(f):
        word, *values = line.rstrip().split(" ")
        words.append(word)
        vectors[i] = values

index = {word: i for i, word in enumerate(words)}
unit = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
print(f"Loaded {n_words} words, {dim} dimensions each")

In [ ]:
#@title most_similar(): finds the closest words to a vector sum 
def most_similar(positive=(), negative=(), topn=5):
    """Words closest (by cosine similarity) to the positive vectors minus the negative ones.

    Each item can be a word or a raw vector.
    """
    def as_unit(x):
        return unit[index[x]] if isinstance(x, str) else x / np.linalg.norm(x)

    query = sum(as_unit(x) for x in positive) - sum(as_unit(x) for x in negative)
    scores = unit @ (query / np.linalg.norm(query))

    # Like gensim, never return the words we were asked about
    exclude = {index[x] for x in [*positive, *negative] if isinstance(x, str)}
    best = np.argsort(-scores)[: topn + len(exclude)]
    return [(words[i], float(scores[i])) for i in best if i not in exclude][:topn]

In [ ]:
# Similar words: every word is a vector, nearby vectors have related meanings
most_similar(positive=["art"], topn=5)

In [ ]:
# The classic: king - man + woman
positive = ['king', 'woman']
negative = ['man']
most_similar(positive=positive, negative=negative, topn=2)

In [ ]:
# Gender clichés: doctor - man + woman
positive = ['doctor', 'woman']
negative = ['man']
most_similar(positive=positive, negative=negative, topn=2)

In [ ]:
# Capital cities: paris - france + italy
positive = ['paris', 'italy']
negative = ['france']
most_similar(positive=positive, negative=negative, topn=3)

In [ ]:
# Leaders:
positive = ['trump', 'russia']
negative = ['usa']
most_similar(positive=positive, negative=negative, topn=3)

In [ ]:
# Nationality: french - france + germany
positive = ['french', 'spain']
negative = ['france']
most_similar(positive=positive, negative=negative, topn=3)

In [ ]:
# Who makes it: painter - painting + music
positive = ['painter', 'book']
negative = ['painting']
most_similar(positive=positive, negative=negative, topn=3)